In [1]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 90)

In [2]:
# Partie 1 - Exploration du corpus

## 1) Chargement des données
df = pd.read_csv("../data/smart_reviews_raw.csv")
df.head(10)

,id_avis,date,source,produit,texte,sentiment,note,langue
0,AV0001,2026-02-27,mobile,Ordinateur NovaBook,"Très bonne expérience, simple et efficace.",positif,4,fr
1,AV0002,2026-01-09,web,SmartPhone X,Très satisfait de mon achat 👍 #avis,positif,4,fr
2,AV0003,2026-07-03,réseaux_sociaux,Écouteurs AirSound,"Produit parfait, rien à signaler.",positif,4,fr
3,AV0004,2026-06-28,sav,SmartWatch Pro,"Produit excellent, je suis très satisfait. !!!",positif,5,fr
4,AV0005,2026-01-24,sav,SmartPhone X,LA BATTERIE TIENT VRAIMENT BIEN ET L'ÉCRAN EST SUPERBE.,positif,5,fr
5,AV0006,2026-07-26,web,Ordinateur NovaBook,Très satisfait de mon achat 👍 😊,positif,5,fr
6,AV0007,2026-05-22,email,Ordinateur NovaBook,SERVICE CLIENT RÉACTIF ET COMMANDE REÇUE RAPIDEMENT.,positif,4,fr
7,AV0008,2026-06-30,web,SmartPhone X,Très satisfait de mon achat 👍,positif,5,fr
8,AV0009,2026-08-10,web,Écouteurs AirSound,Le produit correspond globalement à la description.,neutre,3,fr
9,AV0010,2026-04-01,mobile,Ordinateur NovaBook,@client Livraison rapide et produit conforme à mes attentes.,positif,5,fr


In [3]:
## 2) Combien d'avis contient le dataset ?

print(f"Nombre d'avis : {len(df)}")

Nombre d'avis : 1200


In [4]:
## 3) Combien de colonnes possède-t-il ?

print(f"Nombre de colonnes : {df.shape[1]}")
print(list(df.columns))

Nombre de colonnes : 8
['id_avis', 'date', 'source', 'produit', 'texte', 'sentiment', 'note', 'langue']


In [5]:
# 4) Type de chaque colonne
# Pourquoi : le type d'une colonne décide de ce qu'on peut faire avec
# (une moyenne se calcule sur des nombres, un nettoyage s'applique à du texte).
df.dtypes

# Réponse : toutes les colonnes sont de type texte (object) sauf `note`, qui est un entier (int64).
# La colonne `date` est lue comme du texte : on pourra la convertir avec pd.to_datetime si besoin.

id_avis      object
date         object
source       object
produit      object
texte        object
sentiment    object
note          int64
langue       object
dtype: object

In [6]:
# 5) Existe-t-il des valeurs manquantes ?
# Pourquoi : un texte manquant (NaN) ne peut ni être nettoyé ni transformé en nombres,
# il faudra donc le supprimer avant de continuer.

# Nombre de valeurs manquantes (NaN) dans chaque colonne
print(df.isna().sum())
print()

# Un texte peut aussi être "vide" sans être NaN, s'il ne contient que des espaces
print("Textes composés uniquement d'espaces :", (df["texte"].fillna("x").str.strip() == "").sum())

# Réponse : oui, uniquement dans la colonne `texte` : 5 valeurs manquantes (NaN),
# auxquelles s'ajoute 1 texte composé uniquement d'espaces, soit 6 textes vides au total.
# Les autres colonnes sont complètes.

id_avis      0
date         0
source       0
produit      0
texte        5
sentiment    0
note         0
langue       0
dtype: int64

Textes composés uniquement d'espaces : 1


In [7]:
# 6) Identifier quelques types de texte
# Pourquoi : repérer les problèmes de qualité du corpus pour savoir quoi traiter au nettoyage.

# On remplace les NaN par "" pour que les fonctions de texte ne plantent pas
texte = df["texte"].fillna("")

# Motifs de recherche (expressions régulières), définis une fois pour être réutilisés plus tard
RE_URL     = r"https?://\S+|www\.\S+"                 # http://... ou www....
RE_MENTION = r"@\w+"                                  # @pseudo
RE_HASHTAG = r"#\w+"                                  # #mot
RE_EMOJI   = r"[\U0001F300-\U0001FAFF\u2600-\u27BF]"  # plages Unicode des emojis
RE_PONCT   = r"[!?.,;:]{2,}"                          # ponctuation répétée (!!, ?!, ...)
RE_REPET   = r"([a-zA-ZÀ-ÿ])\1{2,}"                   # même lettre 3 fois d'affilée (superrrr)

# Fonction : vrai si le texte est écrit uniquement en majuscules
def est_majuscules(s):
    lettres = [c for c in s if c.isalpha()]
    return len(lettres) > 3 and all(c.isupper() for c in lettres)

# Un masque par type : une série de vrai/faux, une valeur par avis
masques = {
    "texte vide"           : texte.str.strip() == "",
    "contient une URL"     : texte.str.contains(RE_URL, regex=True),
    "contient une mention" : texte.str.contains(RE_MENTION, regex=True),
    "contient un hashtag"  : texte.str.contains(RE_HASHTAG, regex=True),
    "contient des emojis"  : texte.str.contains(RE_EMOJI, regex=True),
    "ponctuation répétée"  : texte.str.contains(RE_PONCT, regex=True),
    "texte en MAJUSCULES"  : texte.apply(est_majuscules),
    "lettres répétées"     : texte.str.contains(RE_REPET, regex=True),
}

# Texte "normal" = aucun des problèmes ci-dessus
normal = ~pd.concat(masques.values(), axis=1).any(axis=1)
print("Exemple de texte normal :", texte[normal].iloc[0])

# Pour chaque type : nombre d'avis concernés et 3 exemples
for nom, m in masques.items():
    print(f"\n--- {nom} : {m.sum()} avis")
    for ex in texte[m].head(3):
        print("   ", repr(ex))

# Réponse : le corpus contient bien tous ces cas (textes vides, URLs, mentions, hashtags,
# emojis, ponctuation répétée, majuscules, lettres répétées) : il faudra les traiter en Partie 2.

Exemple de texte normal : Très  bonne  expérience,  simple  et  efficace.

--- texte vide : 6 avis
    ''
    ''
    ''

--- contient une URL : 143 avis
    'Produit parfait, rien à signaler. https://example.com/commande/17'
    'Très bonne expérience, simple et efficace. https://example.com/commande/31'
    'Je viens de recevoir le produit, à voir dans le temps. https://example.com/commande/32'

--- contient une mention : 128 avis
    '@client Livraison rapide et produit conforme à mes attentes.'
    '@client Service client réactif et commande reçue rapidement.'
    '@client La qualité est correcte sans être exceptionnelle.'

--- contient un hashtag : 127 avis
    'Très satisfait de mon achat 👍 #avis'
    "La batterie tient vraiment bien et l'écran est superbe. #avis"
    'Produit correct pour son prix. #avis'

--- contient des emojis : 192 avis
    'Très satisfait de mon achat 👍 #avis'
    'Très satisfait de mon achat 👍 😊'
    'Très  satisfait  de  mon  achat  👍'

--- ponctuation rép

C:\Users\dell\AppData\Local\Temp\ipykernel_18304\21312265.py:29: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  "lettres répétées"     : texte.str.contains(RE_REPET, regex=True),


In [8]:
# 7) Mesurer la longueur des textes
# Pourquoi : la longueur (en caractères) permet de repérer les textes trop courts ou trop longs,
# qui peuvent être des anomalies ou apporter peu d'information au modèle.

# Nouvelle colonne "longueur" : nombre de caractères de chaque texte
# (les textes manquants sont comptés pour 0)
df["longueur"] = df["texte"].fillna("").str.len()

# Statistiques descriptives de la longueur
stats = df["longueur"].describe()
print(f"Minimum  : {int(stats['min'])}")
print(f"Maximum  : {int(stats['max'])}")
print(f"Moyenne  : {stats['mean']:.2f}")
print(f"Médiane  : {df['longueur'].median():.0f}")
print(f"Q1 (25%) : {stats['25%']:.0f}")
print(f"Q3 (75%) : {stats['75%']:.0f}")

# Réponse : les avis font en moyenne environ 49 caractères, avec une médiane de 48.
# La moitié des avis se situe entre 37 et 56 caractères (Q1 à Q3).
# Le minimum est 0 (textes vides) et le maximum 636, ce qui est très éloigné du reste.

Minimum  : 0
Maximum  : 636
Moyenne  : 48.96
Médiane  : 48
Q1 (25%) : 37
Q3 (75%) : 56


In [ ]:
# 7) Mesurer la longueur des textes
# Pourquoi : la longueur (en caractères) permet de repérer les textes trop courts ou trop longs,
# qui peuvent être des anomalies ou apporter peu d'information au modèle.

# Nouvelle colonne "longueur" : nombre de caractères de chaque texte
# (les textes manquants sont comptés pour 0)
df["longueur"] = df["texte"].fillna("").str.len()

# Statistiques descriptives de la longueur
stats = df["longueur"].describe()
print(f"Minimum  : {int(stats['min'])}")
print(f"Maximum  : {int(stats['max'])}")
print(f"Moyenne  : {stats['mean']:.2f}")
print(f"Médiane  : {df['longueur'].median():.0f}")
print(f"Q1 (25%) : {stats['25%']:.0f}")
print(f"Q3 (75%) : {stats['75%']:.0f}")

# Réponse : les avis font en moyenne environ 49 caractères, avec une médiane de 48.
# La moitié des avis se situe entre 37 et 56 caractères (Q1 à Q3).
# Le minimum est 0 (textes vides) et le maximum 636, ce qui est très éloigné du reste.